# 01 — Data Loading & Validation
**Project:** Olist E-Commerce — Sales Funnel & RFM Analysis  
**Phase:** 1 of 5  
**Goal:** Load all 9 Olist tables, inspect structure, fix data types, validate relationships.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

DATA_PATH = Path('../data/raw')
print('Data path exists:', DATA_PATH.exists())

## 1.1 — Load all tables

In [ ]:
orders       = pd.read_csv(DATA_PATH / 'olist_orders_dataset.csv')
order_items  = pd.read_csv(DATA_PATH / 'olist_order_items_dataset.csv')
customers    = pd.read_csv(DATA_PATH / 'olist_customers_dataset.csv')
products     = pd.read_csv(DATA_PATH / 'olist_products_dataset.csv')
sellers      = pd.read_csv(DATA_PATH / 'olist_sellers_dataset.csv')
payments     = pd.read_csv(DATA_PATH / 'olist_order_payments_dataset.csv')
reviews      = pd.read_csv(DATA_PATH / 'olist_order_reviews_dataset.csv')
category_t   = pd.read_csv(DATA_PATH / 'product_category_name_translation.csv')
geolocation  = pd.read_csv(DATA_PATH / 'olist_geolocation_dataset.csv')

tables = {
    'orders':      orders,
    'order_items': order_items,
    'customers':   customers,
    'products':    products,
    'sellers':     sellers,
    'payments':    payments,
    'reviews':     reviews,
    'category_t':  category_t,
    'geolocation': geolocation,
}

print(f"{'Table':<15} {'Rows':>10} {'Cols':>6}")
print('-' * 35)
for name, df in tables.items():
    print(f"{name:<15} {df.shape[0]:>10,} {df.shape[1]:>6}")

## 1.2 — Fix datetime columns

In [ ]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

reviews['review_creation_date']       = pd.to_datetime(reviews['review_creation_date'])
reviews['review_answer_timestamp']    = pd.to_datetime(reviews['review_answer_timestamp'])
order_items['shipping_limit_date']    = pd.to_datetime(order_items['shipping_limit_date'])

print('Date range of orders:')
print('  From:', orders['order_purchase_timestamp'].min())
print('  To:  ', orders['order_purchase_timestamp'].max())

## 1.3 — Order status breakdown

In [ ]:
status_counts = orders['order_status'].value_counts()
print(status_counts)

fig, ax = plt.subplots(figsize=(8, 4))
status_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Order status distribution', fontsize=13)
ax.set_xlabel('')
ax.set_ylabel('Count')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../reports/01_order_status.png', dpi=150)
plt.show()

## 1.4 — Null audit across all tables

In [ ]:
print('=== NULL AUDIT ===')
for name, df in tables.items():
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if not nulls.empty:
        print(f'\n{name}:')
        for col, n in nulls.items():
            print(f'  {col}: {n:,} nulls ({n/len(df)*100:.1f}%)')
print('\n✅ Null audit complete')

## 1.5 — Validate foreign key relationships

In [ ]:
# Check that order_items and payments reference valid order_ids
valid_orders = set(orders['order_id'])

items_orphaned    = order_items[~order_items['order_id'].isin(valid_orders)]
payments_orphaned = payments[~payments['order_id'].isin(valid_orders)]
reviews_orphaned  = reviews[~reviews['order_id'].isin(valid_orders)]

print(f'Orphaned order_items rows : {len(items_orphaned):,}')
print(f'Orphaned payment rows     : {len(payments_orphaned):,}')
print(f'Orphaned review rows      : {len(reviews_orphaned):,}')
print('\n✅ FK validation complete')

## 1.6 — Save clean orders to processed/

In [ ]:
# Keep only delivered orders for main analysis (filter out cancelled/unavailable)
orders_clean = orders[orders['order_status'] == 'delivered'].copy()
orders_clean['delivery_delay_days'] = (
    orders_clean['order_delivered_customer_date'] -
    orders_clean['order_estimated_delivery_date']
).dt.days

print(f'Delivered orders: {len(orders_clean):,} ({len(orders_clean)/len(orders)*100:.1f}% of total)')

orders_clean.to_csv('../data/processed/orders_clean.csv', index=False)
print('✅ Saved to data/processed/orders_clean.csv')

---
## ✅ Phase 1 Summary

| Check | Result |
|---|---|
| All 9 tables loaded | ✅ |
| Datetime columns fixed | ✅ |
| Nulls identified | ✅ |
| FK relationships validated | ✅ |
| Clean orders saved | ✅ |

**Next:** `02_eda.ipynb` — Exploratory data analysis